<a href="https://colab.research.google.com/github/NIHAL-PM/AI-AGENT-VISION-AND-VOICE/blob/main/Complete_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# -*- coding: utf-8 -*-
"""
End-to-End Stuttering Detection Pipeline (SEP-28k)
==================================================
A single-file solution that:
1. Auto-downloads dataset labels (SEP-28k).
2. Downloads raw podcast audio.
3. Clips and sorts audio into 'fluent' vs 'stuttering'.
4. Trains a CNN-LSTM model on the real data.
5. Demonstrates the therapy logic.

WARNING: The full dataset is large (20GB+).
By default, DEMO_MODE is True (downloads only 5 episodes).
Set DEMO_MODE = False to train on the full dataset.
"""

import os
import pathlib
import subprocess
import urllib.request
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from scipy.io import wavfile
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Input
from sklearn.model_selection import train_test_split
import random

# ==========================================
# 1. GLOBAL CONFIGURATION
# ==========================================
DEMO_MODE = False  # ⚠️ Set to False for REAL results (takes longer)
LANGUAGE = 'en'   # 'en', 'es', 'hi', 'fr'
EPOCHS = 20       # Increased for better learning

# Paths
BASE_DIR = "/content"
RAW_WAV_DIR = os.path.join(BASE_DIR, "sep28k_raw_wavs")
DATASET_OUTPUT = os.path.join(BASE_DIR, "dataset")
EPISODES_CSV = os.path.join(BASE_DIR, "SEP-28k_episodes.csv")
LABELS_CSV = os.path.join(BASE_DIR, "SEP-28k_labels.csv")

# URLs for Metadata
URL_EPISODES = "https://raw.githubusercontent.com/apple/ml-stuttering-events-dataset/main/SEP-28k_episodes.csv"
URL_LABELS = "https://raw.githubusercontent.com/apple/ml-stuttering-events-dataset/main/SEP-28k_labels.csv"

# Audio Params for Model
SAMPLE_RATE = 16000
DURATION = 3
N_MFCC = 40
MAX_LEN = 130

# ==========================================
# 2. UTILITIES & SETUP
# ==========================================
def setup_environment():
    print("\n[1/5] Setting up Environment & Fetching Metadata...")

    # Download CSVs if missing
    if not os.path.exists(EPISODES_CSV):
        print(f"   Downloading {EPISODES_CSV}...")
        try:
            urllib.request.urlretrieve(URL_EPISODES, EPISODES_CSV)
        except Exception as e:
            print(f"   ❌ Error downloading episodes CSV: {e}")

    if not os.path.exists(LABELS_CSV):
        print(f"   Downloading {LABELS_CSV}...")
        try:
            urllib.request.urlretrieve(URL_LABELS, LABELS_CSV)
        except Exception as e:
            print(f"   ❌ Error downloading labels CSV: {e}")

    # Install dependencies if missing
    try:
        import soundfile
    except ImportError:
        print("   Installing soundfile...")
        subprocess.check_call(["pip", "install", "soundfile"])

    print("   ✅ Setup Complete.")

# ==========================================
# 3. DOWNLOADER (Raw Podcasts)
# ==========================================
def download_raw_audio():
    print("\n[2/5] Downloading Audio Episodes...")

    if not os.path.exists(EPISODES_CSV):
        print("❌ Critical Error: CSV not found.")
        return

    # --- FIX: Handle No-Header CSV ---
    try:
        # SEP-28k_episodes.csv has NO header.
        # Columns are: ShowName, EpTitle, Url, Show, EpId
        df = pd.read_csv(EPISODES_CSV, sep=r',\s+', engine='python', header=None)

        # Manually assign correct column names
        df.columns = ['ShowName', 'EpTitle', 'Url', 'Show', 'EpId']

        # Clean up data (remove quotes if any, strip spaces)
        df['Show'] = df['Show'].astype(str).str.strip()
        df['EpId'] = df['EpId'].astype(str).str.strip()

    except Exception as e:
        print(f"❌ Error reading CSV file: {e}")
        return
    # -------------------------------

    if DEMO_MODE:
        print("   ⚠️ DEMO MODE ACTIVE: Downloading only first 5 episodes.")
        df = df.head(5)

    os.makedirs(RAW_WAV_DIR, exist_ok=True)

    for index, row in df.iterrows():
        try:
            show_abrev = row['Show']
            ep_idx = row['EpId']
            episode_url = row['Url']
        except KeyError as e:
            print(f"   ❌ Missing column in row {index}: {e}")
            continue

        episode_dir = pathlib.Path(f"{RAW_WAV_DIR}/{show_abrev}/")
        os.makedirs(episode_dir, exist_ok=True)
        wav_path = pathlib.Path(f"{episode_dir}/{ep_idx}.wav")

        if os.path.exists(wav_path):
            print(f"   Skipping {show_abrev} {ep_idx} (Exists)")
            continue

        print(f"   Processing {show_abrev} Episode {ep_idx}...")

        # Download to temp file
        temp_audio = pathlib.Path(f"{episode_dir}/temp_audio")
        # -q for quiet, -O for output file
        cmd_download = f'wget -O "{temp_audio}" "{episode_url}" -q'

        try:
            subprocess.call(cmd_download, shell=True)

            # Check if file exists and has size
            if not os.path.exists(temp_audio) or os.path.getsize(temp_audio) < 1000:
                print(f"      ⚠️ Download failed or file too small: {episode_url}")
                if os.path.exists(temp_audio): os.remove(temp_audio)
                continue

            # Convert to 16khz mono wav using ffmpeg
            cmd_convert = f'ffmpeg -y -i "{temp_audio}" -ac 1 -ar 16000 "{wav_path}" -hide_banner -loglevel error'
            subprocess.call(cmd_convert, shell=True)

            if os.path.exists(temp_audio):
                os.remove(temp_audio)

        except Exception as e:
            print(f"      ❌ Failed: {e}")

    print("   ✅ Audio Download Phase Complete.")

# ==========================================
# 4. CLIPPER (Cut & Sort)
# ==========================================
def clip_and_sort_audio():
    print("\n[3/5] Clipping and Sorting into Fluent/Stuttering...")

    if not os.path.exists(LABELS_CSV):
        print("❌ Labels CSV not found.")
        return

    # Read Labels CSV
    try:
        df = pd.read_csv(LABELS_CSV, dtype={"EpId": str})
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"❌ Error reading labels CSV: {e}")
        return

    df = df.sort_values(by=['Show', 'EpId', 'Start'])

    os.makedirs(f"{DATASET_OUTPUT}/fluent", exist_ok=True)
    os.makedirs(f"{DATASET_OUTPUT}/stuttering", exist_ok=True)

    current_wav_path = ""
    audio_data = None
    sample_rate = 0

    processed_count = 0

    # Iterate through labels
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing Clips"):
        show = row['Show']
        ep_id = str(row['EpId']).strip()
        clip_id = str(row['ClipId'])
        start = int(row['Start'])
        stop = int(row['Stop'])

        wav_path = f"{RAW_WAV_DIR}/{show}/{ep_id}.wav"

        # Optimization: Load file only if changed
        if wav_path != current_wav_path:
            if not os.path.exists(wav_path):
                continue

            try:
                sample_rate, audio_data = wavfile.read(wav_path)
                current_wav_path = wav_path
            except Exception as e:
                continue

        # Classify based on labels
        try:
            is_fluent = row['NoStutteredWords'] >= 2
        except KeyError:
            is_fluent = True

        folder = "fluent" if is_fluent else "stuttering"
        filename = f"{show}_{ep_id}_{clip_id}.wav"
        output_path = f"{DATASET_OUTPUT}/{folder}/{filename}"

        if not os.path.exists(output_path):
            try:
                if audio_data is not None:
                    clip_data = audio_data[start:stop]
                    wavfile.write(output_path, sample_rate, clip_data)
                    processed_count += 1
            except Exception:
                pass

    print(f"   ✅ Clipping Complete. {processed_count} clips generated.")

# ==========================================
# 5. MODEL TRAINING
# ==========================================
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)

        if mfccs.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :MAX_LEN]
        return mfccs.T
    except Exception as e:
        return None

def run_training():
    print("\n[4/5] Training Model...")

    X = []
    y = []

    # Load Data
    categories = {'fluent': 0, 'stuttering': 1}
    valid_data_found = False

    for category, label in categories.items():
        folder_path = os.path.join(DATASET_OUTPUT, category)
        if not os.path.exists(folder_path): continue

        files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
        print(f"   Loading {len(files)} {category} samples...")

        for file in files:
            feat = extract_features(os.path.join(folder_path, file))
            if feat is not None:
                X.append(feat)
                y.append(label)
                valid_data_found = True

    if not valid_data_found:
        print("❌ No training data found. Did the download/clipping work?")
        return None

    X = np.array(X)
    y = np.array(y)

    if len(X) < 5:
        print("⚠️ Not enough data to train (need at least 5 samples).")
        return None

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build Model
    model = Sequential()
    # --- FIX: Using explicit Input layer to avoid warning ---
    model.add(Input(shape=(MAX_LEN, N_MFCC)))
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    # -------------------------------------------------------
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(64))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Train
    print(f"   Training for {EPOCHS} epochs...")
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, validation_data=(X_test, y_test))
    print("   ✅ Model Trained.")
    return model

# ==========================================
# 6. THERAPY LOGIC & EXECUTION
# ==========================================
def therapy_feedback(prediction_prob, lang='en'):
    strategies = {
        'en': ["Take a deep breath.", "Easy Onset: start gently.", "Slow down."],
        'es': ["Respira profundo.", "Inicio Suave.", "Reduce la velocidad."],
        'hi': ["Gehri saans lein.", "Dheere bole.", "Shuruat aaram se karein."],
        'fr': ["Respirez profondément.", "Commencez doucement.", "Ralentissez."]
    }
    msgs = strategies.get(lang, strategies['en'])

    if prediction_prob > 0.5:
        return f"⚠️ Detected ({prediction_prob:.2f}) -> 🤖 BOT: {random.choice(msgs)}"
    return f"✅ Fluent ({prediction_prob:.2f})"

def main():
    setup_environment()
    download_raw_audio()
    clip_and_sort_audio()
    model = run_training()

    if model:
        print("\n[5/5] Running Inference Demo on Test Sample...")
        fake_input = np.random.normal(0, 1, SAMPLE_RATE*DURATION)
        sf.write('temp_demo.wav', fake_input, SAMPLE_RATE)

        feat = extract_features('temp_demo.wav')
        if feat is not None:
            feat = np.expand_dims(feat, axis=0)
            pred = model.predict(feat)[0][0]
            print(f"\nRESULT: {therapy_feedback(pred, lang=LANGUAGE)}")

        if os.path.exists('temp_demo.wav'):
            os.remove('temp_demo.wav')

if __name__ == "__main__":
    main()


[1/5] Setting up Environment & Fetching Metadata...
   ✅ Setup Complete.

[2/5] Downloading Audio Episodes...
   Skipping HeStutters 0 (Exists)
   Skipping HeStutters 1 (Exists)
   Skipping HeStutters 2 (Exists)
   Skipping HeStutters 3 (Exists)
   Skipping HeStutters 4 (Exists)
   Processing HeStutters Episode 5...
   Processing HeStutters Episode 6...
   Processing HeStutters Episode 7...
   Processing HeStutters Episode 8...
   Processing HeStutters Episode 9...
   Processing HeStutters Episode 10...
   Processing HeStutters Episode 11...
   Processing HeStutters Episode 12...
   Processing HeStutters Episode 13...
   Processing HeStutters Episode 14...
   Processing HeStutters Episode 15...
   Processing HeStutters Episode 16...
   Processing HeStutters Episode 17...
   Processing HeStutters Episode 18...
   Processing HeStutters Episode 19...
   Processing HeStutters Episode 20...
   Processing HeStutters Episode 21...
   Processing HeStutters Episode 22...
   Processing HeStutte

Processing Clips: 100%|██████████| 28177/28177 [01:54<00:00, 245.85it/s]


   ✅ Clipping Complete. 20297 clips generated.

[4/5] Training Model...
   Loading 11537 fluent samples...
   Loading 9369 stuttering samples...
   Training for 20 epochs...
Epoch 1/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.5357 - loss: 0.6908 - val_accuracy: 0.5579 - val_loss: 0.6877
Epoch 2/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.5647 - loss: 0.6820 - val_accuracy: 0.5748 - val_loss: 0.6799
Epoch 3/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5703 - loss: 0.6790 - val_accuracy: 0.5600 - val_loss: 0.6785
Epoch 4/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.5890 - loss: 0.6734 - val_accuracy: 0.5770 - val_loss: 0.6762
Epoch 5/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5923 - loss: 0.6718 - val_accuracy: 0.5815 - val_loss: 0.6733
Epoch 6/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.5932 - loss: 0.6659 - val_accuracy: 0.5892 - val_loss: 0.6695
Epoch 7/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step

In [9]:
# -*- coding: utf-8 -*-
"""
End-to-End Stuttering Detection Pipeline (SEP-28k)
==================================================
A single-file solution that:
1. Auto-downloads dataset labels (SEP-28k).
2. Downloads raw podcast audio.
3. Clips and sorts audio into 'fluent' vs 'stuttering'.
4. Trains a CNN-LSTM model on the real data (with feature enrichment).
5. Saves the model permanently to Google Drive.
6. Demonstrates the therapy logic.

NOTE ON ACCURACY: Achieving >90% accuracy on real-world stuttering detection
(SEP-28k) is highly challenging, even for state-of-the-art models. This script
uses major feature engineering (120 MFCCs) to optimize performance with the
CNN-LSTM architecture.
"""

import os
import pathlib
import subprocess
import urllib.request
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from scipy.io import wavfile
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Input
from sklearn.model_selection import train_test_split
import random

# ==========================================
# 1. GLOBAL CONFIGURATION
# ==========================================
DEMO_MODE = False  # Set to False to download ALL episodes (May take a long time)
LANGUAGE = 'en'   # 'en', 'es', 'hi', 'fr'
EPOCHS = 20       # Training epochs

# Paths
BASE_DIR = "/content"
RAW_WAV_DIR = os.path.join(BASE_DIR, "sep28k_raw_wavs")
DATASET_OUTPUT = os.path.join(BASE_DIR, "dataset")
EPISODES_CSV = os.path.join(BASE_DIR, "SEP-28k_episodes.csv")
LABELS_CSV = os.path.join(BASE_DIR, "SEP-28k_labels.csv")

# Google Drive Path for Permanent Saving
# IMPORTANT: You MUST mount your Google Drive before running this script
# Use the following placeholder path. Ensure 'StutteringProject' exists in your Drive.
DRIVE_SAVE_PATH = "/content/gdrive/MyDrive/StutteringProject/stutter_detector_model.h5"

# If Drive is not mounted, the model will be saved locally (and will be lost on session end)
MODEL_OUTPUT_PATH = DRIVE_SAVE_PATH

# URLs for Metadata
URL_EPISODES = "https://raw.githubusercontent.com/apple/ml-stuttering-events-dataset/main/SEP-28k_episodes.csv"
URL_LABELS = "https://raw.githubusercontent.com/apple/ml-stuttering-events-dataset/main/SEP-28k_labels.csv"

# Audio Params for Model
SAMPLE_RATE = 16000
DURATION = 3
N_MFCC = 40   # Base MFCC count (Total features = N_MFCC * 3 = 120)
MAX_LEN = 130

# ==========================================
# 2. UTILITIES & SETUP
# ==========================================
def setup_environment():
    print("\n[1/5] Setting up Environment & Fetching Metadata...")

    # Download CSVs if missing
    if not os.path.exists(EPISODES_CSV):
        print(f"   Downloading {EPISODES_CSV}...")
        try:
            urllib.request.urlretrieve(URL_EPISODES, EPISODES_CSV)
        except Exception as e:
            print(f"   ❌ Error downloading episodes CSV: {e}")

    if not os.path.exists(LABELS_CSV):
        print(f"   Downloading {LABELS_CSV}...")
        try:
            urllib.request.urlretrieve(URL_LABELS, LABELS_CSV)
        except Exception as e:
            print(f"   ❌ Error downloading labels CSV: {e}")

    # Install dependencies if missing
    try:
        import soundfile
    except ImportError:
        print("   Installing soundfile...")
        subprocess.check_call(["pip", "install", "soundfile"])

    # Check if Google Drive is mounted for saving
    if not os.path.exists("/content/gdrive"):
        print("   ⚠️ Google Drive not mounted. Model will be saved locally (temporary).")
    else:
        # Create the Drive directory if it doesn't exist
        drive_path_root = os.path.dirname(DRIVE_SAVE_PATH)
        os.makedirs(drive_path_root, exist_ok=True)
        print(f"   ✅ Drive mounted. Model will be saved to: {DRIVE_SAVE_PATH}")


    print("   ✅ Setup Complete.")

# ==========================================
# 3. DOWNLOADER (Raw Podcasts)
# ==========================================
def download_raw_audio():
    print("\n[2/5] Downloading Audio Episodes...")

    if not os.path.exists(EPISODES_CSV):
        print("❌ Critical Error: CSV not found.")
        return

    # --- FIX: Handle No-Header CSV ---
    try:
        # SEP-28k_episodes.csv has NO header.
        # Columns are: ShowName, EpTitle, Url, Show, EpId
        df = pd.read_csv(EPISODES_CSV, sep=r',\s+', engine='python', header=None)

        # Manually assign correct column names
        df.columns = ['ShowName', 'EpTitle', 'Url', 'Show', 'EpId']

        # Clean up data (remove quotes if any, strip spaces)
        df['Show'] = df['Show'].astype(str).str.strip()
        df['EpId'] = df['EpId'].astype(str).str.strip()

    except Exception as e:
        print(f"❌ Error reading CSV file: {e}")
        return
    # -------------------------------

    if DEMO_MODE:
        print("   ⚠️ DEMO MODE ACTIVE: Downloading only first 5 episodes.")
        df = df.head(5)
    else:
        # Check if most files already exist before printing "FULL DOWNLOAD" warning
        total_episodes = len(df)
        downloaded_count = 0
        for index, row in df.iterrows():
            wav_path = pathlib.Path(f"{RAW_WAV_DIR}/{row['Show']}/{row['EpId']}.wav")
            if os.path.exists(wav_path):
                downloaded_count += 1

        if downloaded_count < total_episodes:
            print("   ✅ FULL DOWNLOAD MODE ACTIVE (May take a long time and several GB of space).")
        else:
            print("   ✅ All episodes previously downloaded. Skipping download phase.")

    os.makedirs(RAW_WAV_DIR, exist_ok=True)

    for index, row in df.iterrows():
        try:
            show_abrev = row['Show']
            ep_idx = row['EpId']
            episode_url = row['Url']
        except KeyError as e:
            print(f"   ❌ Missing column in row {index}: {e}")
            continue

        episode_dir = pathlib.Path(f"{RAW_WAV_DIR}/{show_abrev}/")
        os.makedirs(episode_dir, exist_ok=True)
        wav_path = pathlib.Path(f"{episode_dir}/{ep_idx}.wav")

        if os.path.exists(wav_path):
            print(f"   Skipping {show_abrev} {ep_idx} (Exists)")
            continue

        print(f"   Processing {show_abrev} Episode {ep_idx}...")

        # Download to temp file
        temp_audio = pathlib.Path(f"{episode_dir}/temp_audio")
        # -q for quiet, -O for output file
        cmd_download = f'wget -O "{temp_audio}" "{episode_url}" -q'

        try:
            subprocess.call(cmd_download, shell=True)

            # Check if file exists and has size
            if not os.path.exists(temp_audio) or os.path.getsize(temp_audio) < 1000:
                # Clean up if download failed/file too small
                print(f"      ⚠️ Download failed or file too small: {episode_url}")
                if os.path.exists(temp_audio): os.remove(temp_audio)
                continue

            # Convert to 16khz mono wav using ffmpeg
            cmd_convert = f'ffmpeg -y -i "{temp_audio}" -ac 1 -ar 16000 "{wav_path}" -hide_banner -loglevel error'
            subprocess.call(cmd_convert, shell=True)

            if os.path.exists(temp_audio):
                os.remove(temp_audio)

        except Exception as e:
            print(f"      ❌ Failed: {e}")

    print("   ✅ Audio Download Phase Complete.")

# ==========================================
# 4. CLIPPER (Cut & Sort)
# ==========================================
def clip_and_sort_audio():
    print("\n[3/5] Clipping and Sorting into Fluent/Stuttering...")

    if not os.path.exists(LABELS_CSV):
        print("❌ Labels CSV not found.")
        return

    # Read Labels CSV
    try:
        df = pd.read_csv(LABELS_CSV, dtype={"EpId": str})
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"❌ Error reading labels CSV: {e}")
        return

    df = df.sort_values(by=['Show', 'EpId', 'Start'])

    os.makedirs(f"{DATASET_OUTPUT}/fluent", exist_ok=True)
    os.makedirs(f"{DATASET_OUTPUT}/stuttering", exist_ok=True)

    current_wav_path = ""
    audio_data = None
    sample_rate = 0

    processed_count = 0

    # Iterate through labels
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing Clips"):
        show = row['Show']
        ep_id = str(row['EpId']).strip()
        clip_id = str(row['ClipId'])
        start = int(row['Start'])
        stop = int(row['Stop'])

        wav_path = f"{RAW_WAV_DIR}/{show}/{ep_id}.wav"

        # Optimization: Load file only if changed
        if wav_path != current_wav_path:
            if not os.path.exists(wav_path):
                continue

            try:
                sample_rate, audio_data = wavfile.read(wav_path)
                current_wav_path = wav_path
            except Exception as e:
                continue

        # Classify based on labels
        try:
            # Stuttering is any clip where less than 2/3 annotators said "No Stutter"
            is_fluent = row['NoStutteredWords'] >= 2
        except KeyError:
            is_fluent = True

        folder = "fluent" if is_fluent else "stuttering"
        filename = f"{show}_{ep_id}_{clip_id}.wav"
        output_path = f"{DATASET_OUTPUT}/{folder}/{filename}"

        if not os.path.exists(output_path):
            try:
                if audio_data is not None:
                    # Clip data slicing
                    clip_data = audio_data[start:stop]
                    wavfile.write(output_path, sample_rate, clip_data)
                    processed_count += 1
            except Exception:
                pass

    print(f"   ✅ Clipping Complete. {processed_count} clips generated.")

# ==========================================
# 5. MODEL TRAINING
# ==========================================
def extract_features(file_path):
    """
    Extracts MFCCs, Delta MFCCs, and Delta-Delta MFCCs (120 features total)
    to capture acoustic information and its rate of change (critical for stutters).

    """
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)

        # 1. Base MFCCs (40 features)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)

        # 2. Delta MFCCs (40 features) - Rate of change
        mfccs_delta = librosa.feature.delta(mfccs)

        # 3. Delta-Delta MFCCs (40 features) - Acceleration of change
        mfccs_delta2 = librosa.feature.delta(mfccs, order=2)

        # Combine all 120 features vertically (40 rows * 3)
        features = np.vstack([mfccs, mfccs_delta, mfccs_delta2])

        # Pad or truncate to MAX_LEN
        if features.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - features.shape[1]
            features = np.pad(features, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            features = features[:, :MAX_LEN]

        # Transpose to shape (Time, Features) -> (130, 120)
        return features.T
    except Exception as e:
        return None

def run_training():
    print("\n[4/5] Training Model...")

    X = []
    y = []

    # Load Data
    categories = {'fluent': 0, 'stuttering': 1}
    valid_data_found = False

    for category, label in categories.items():
        folder_path = os.path.join(DATASET_OUTPUT, category)
        if not os.path.exists(folder_path): continue

        files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
        print(f"   Loading {len(files)} {category} samples...")

        for file in files:
            feat = extract_features(os.path.join(folder_path, file))
            if feat is not None:
                X.append(feat)
                y.append(label)
                valid_data_found = True

    if not valid_data_found:
        print("❌ No training data found. Did the download/clipping work?")
        return None

    X = np.array(X)
    y = np.array(y)

    if len(X) < 5:
        print("⚠️ Not enough data to train (need at least 5 samples).")
        return None

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build Model
    # Input shape is (MAX_LEN, 120 features)
    model = Sequential()
    model.add(Input(shape=(MAX_LEN, N_MFCC * 3)))
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(64))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Train
    print(f"   Training for {EPOCHS} epochs...")
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, validation_data=(X_test, y_test))

    # --- Saving the model to disk (Permanent) ---
    try:
        model.save(MODEL_OUTPUT_PATH)
        print(f"   ✅ Model Trained and saved permanently to: {MODEL_OUTPUT_PATH}")
    except Exception as e:
        print(f"   ❌ Error saving model to Drive ({e}). Ensure Google Drive is mounted.")

    return model

# ==========================================
# 6. THERAPY LOGIC & EXECUTION
# ==========================================
def therapy_feedback(prediction_prob, lang='en'):
    strategies = {
        'en': ["Take a deep breath.", "Easy Onset: start gently.", "Slow down."],
        'es': ["Respira profundo.", "Inicio Suave.", "Reduce la velocidad."],
        'hi': ["Gehri saans lein.", "Dheere bole.", "Shuruat aaram se karein."],
        'fr': ["Respirez profondément.", "Commencez doucement.", "Ralentissez."]
    }
    msgs = strategies.get(lang, strategies['en'])

    if prediction_prob > 0.5:
        return f"⚠️ Detected ({prediction_prob:.2f}) -> 🤖 BOT: {random.choice(msgs)}"
    return f"✅ Fluent ({prediction_prob:.2f})"

def main():
    setup_environment()
    download_raw_audio()
    clip_and_sort_audio()
    model = run_training()

    if model:
        print("\n[5/5] Running Inference Demo on Test Sample...")
        fake_input = np.random.normal(0, 1, SAMPLE_RATE*DURATION)
        sf.write('temp_demo.wav', fake_input, SAMPLE_RATE)

        feat = extract_features('temp_demo.wav')
        if feat is not None:
            feat = np.expand_dims(feat, axis=0)
            pred = model.predict(feat)[0][0]
            print(f"\nRESULT: {therapy_feedback(pred, lang=LANGUAGE)}")

        if os.path.exists('temp_demo.wav'):
            os.remove('temp_demo.wav')

if __name__ == "__main__":
    main()


[1/5] Setting up Environment & Fetching Metadata...
   ⚠️ Google Drive not mounted. Model will be saved locally (temporary).
   ✅ Setup Complete.

[2/5] Downloading Audio Episodes...
   ✅ FULL DOWNLOAD MODE ACTIVE (May take a long time and several GB of space).
   Processing HeStutters Episode 0...
   Processing HeStutters Episode 1...
   Processing HeStutters Episode 2...
   Processing HeStutters Episode 3...
   Processing HeStutters Episode 4...
   Processing HeStutters Episode 5...
   Processing HeStutters Episode 6...
   Processing HeStutters Episode 7...
   Processing HeStutters Episode 8...
   Processing HeStutters Episode 9...
   Processing HeStutters Episode 10...
   Processing HeStutters Episode 11...
   Processing HeStutters Episode 12...
   Processing HeStutters Episode 13...
   Processing HeStutters Episode 14...
   Processing HeStutters Episode 15...
   Processing HeStutters Episode 16...
   Processing HeStutters Episode 17...
   Processing HeStutters Episode 18...
   Pro

Processing Clips: 100%|██████████| 28177/28177 [01:36<00:00, 290.83it/s]


   ✅ Clipping Complete. 20906 clips generated.

[4/5] Training Model...
   Loading 11537 fluent samples...
   Loading 9369 stuttering samples...
   Training for 20 epochs...
Epoch 1/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.5461 - loss: 0.6889 - val_accuracy: 0.5450 - val_loss: 0.6858
Epoch 2/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5522 - loss: 0.6826 - val_accuracy: 0.5679 - val_loss: 0.6867
Epoch 3/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.5629 - loss: 0.6775 - val_accuracy: 0.5689 - val_loss: 0.6800
Epoch 4/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5893 - loss: 0.6732 - val_accuracy: 0.5854 - val_loss: 0.6712
Epoch 5/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.5961 - loss: 0.6661 - val_accuracy: 0.5835 - val_loss: 0.6724
Epoch 6/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5912 - loss: 0.6678 - val_accuracy: 0.5863 - val_loss: 0.6697
Epoch 7/20
523/523 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/ste

   ✅ Model Trained and saved permanently to: /content/gdrive/MyDrive/StutteringProject/stutter_detector_model.h5

[5/5] Running Inference Demo on Test Sample...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step

RESULT: ✅ Fluent (0.16)
